In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [5]:
import os
import shutil
import pandas as pd

DRIVE = "/content/drive/MyDrive"
SHEET_SRC = f"{DRIVE}/evidence_relevance_check.csv"
ANN_DIR = f"{DRIVE}/BFND-data/data/annotations"
SHEET_DST = f"{ANN_DIR}/evidence_relevance_check.csv"

assert os.path.exists(SHEET_SRC), "Sheet not found in MyDrive root"
os.makedirs(ANN_DIR, exist_ok=True)

sheet = pd.read_csv(SHEET_SRC, dtype=str, keep_default_na=False)
assert "relevant_0_or_1" in sheet.columns, "Label column missing"
blank = (sheet["relevant_0_or_1"].str.strip() == "").sum()
print("Rows:", len(sheet), "| blank labels:", blank)

if os.path.exists(SHEET_DST):
    backup = f"{ANN_DIR}/evidence_relevance_check_before_copy.csv"
    if not os.path.exists(backup):
        shutil.copy2(SHEET_DST, backup)
        print("Old sheet backed up to:", backup)

shutil.copy2(SHEET_SRC, SHEET_DST)
print("Copied to:", SHEET_DST)

Rows: 40 | blank labels: 40
Old sheet backed up to: /content/drive/MyDrive/BFND-data/data/annotations/evidence_relevance_check_before_copy.csv
Copied to: /content/drive/MyDrive/BFND-data/data/annotations/evidence_relevance_check.csv


In [6]:
import os
import zipfile

DRIVE = "/content/drive/MyDrive"
PROJECT = "/content/Bangla-Multimodal-FakeNews"
BASE_ZIP = f"{DRIVE}/Bangla-Multimodal-FakeNews.zip"
UPDATE_ZIP = f"{DRIVE}/BFND_evidence_update.zip"

assert os.path.exists(BASE_ZIP), "Base zip not found"
assert os.path.exists(UPDATE_ZIP), "Update zip not found"

with zipfile.ZipFile(BASE_ZIP) as z:
    names = z.namelist()
top_level = {n.split("/")[0] for n in names if n}
nested = top_level == {"Bangla-Multimodal-FakeNews"}
print("Top-level in base zip:", sorted(top_level)[:10], "| nested:", nested)

if not os.path.isdir(os.path.join(PROJECT, "app")):
    os.makedirs(PROJECT, exist_ok=True)
    with zipfile.ZipFile(BASE_ZIP) as z:
        z.extractall("/content" if nested else PROJECT)
    print("Base project extracted")
else:
    print("Project folder exists; base zip skipped")

# The base must contain the OCR code, otherwise it is an old version
with open(os.path.join(PROJECT, "src/ocr.py"), encoding="utf-8") as f:
    assert "def ocr_pil" in f.read(), (
        "Base zip is OLD (no ocr_pil). Stop here and tell me; do not continue."
    )

with zipfile.ZipFile(UPDATE_ZIP) as z:
    z.extractall(PROJECT)
print("Evidence update applied")

Top-level in base zip: ['Bangla-Multimodal-FakeNews'] | nested: True
Base project extracted
Evidence update applied


In [8]:
import os
import pandas as pd

DRIVE = "/content/drive/MyDrive"
ANN = f"{DRIVE}/BFND-data/data/annotations"
candidates = {
    "annotations (used by script 09)": f"{ANN}/evidence_relevance_check.csv",
    "annotations backup (old copy)": f"{ANN}/evidence_relevance_check_before_copy.csv",
    "MyDrive root": f"{DRIVE}/evidence_relevance_check.csv",
}

for label, path in candidates.items():
    if not os.path.exists(path):
        print(f"{label}: NOT FOUND")
        continue
    df = pd.read_csv(path, dtype=str, keep_default_na=False)
    vals = df["relevant_0_or_1"].str.strip()
    print(f"{label}: rows={len(df)}, labelled 0/1={vals.isin(['0', '1']).sum()}, "
          f"blank={(vals == '').sum()}, ones={(vals == '1').sum()}")

annotations (used by script 09): rows=40, labelled 0/1=0, blank=40, ones=0
annotations backup (old copy): rows=40, labelled 0/1=40, blank=0, ones=13
MyDrive root: rows=40, labelled 0/1=0, blank=40, ones=0


In [11]:
import json
with open("/content/drive/MyDrive/BFND-data/results/evidence_gate.json", encoding="utf-8") as f:
    print(json.load(f))

{'check': 'evidence relevance (Wikipedia passage vs post claim)', 'threshold': 0.6, 'threshold_source': 'D-4', 'n': 40, 'relevant': 13, 'relevance_share': 0.325, 'status': 'FAIL', 'sheet': 'evidence_relevance_check.csv', 'note': 'Evidence is shown in the app but not used in the prediction unless the gate passes and an E5 validation gain is recorded (D-5).'}


In [3]:
!pip install easyocr

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 35.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 972.1/972.1 kB 29.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [easyocr]


In [6]:
%cd /content/Bangla-Multimodal-FakeNews
from app.capstone_app import load_runtime, build_ui
load_runtime()
demo = build_ui()
demo.launch()

/content/Bangla-Multimodal-FakeNews


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] ElectraModel LOAD REPORT from: csebuetnlp/banglabert
Key                                               | Status     |  | 
--------------------------------------------------+------------+--+-
discriminator_predictions.dense_prediction.weight | UNEXPECTED |  | 
discriminator_predictions.dense.weight            | UNEXPECTED |  | 
discriminator_predictions.dense.bias              | UNEXPECTED |  | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://fbd3960d4b8db52dfd.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [1]:
import os
import zipfile

PROJECT = "/content/Bangla-Multimodal-FakeNews"
ZIPS = [
    "/content/drive/MyDrive/Bangla-Multimodal-FakeNews.zip",   # base
    "/content/drive/MyDrive/BFND_evidence_update.zip",         # evidence
    "/content/drive/MyDrive/BFND_ocr_filter_update.zip",       # OCR filter, last
]

for path in ZIPS:
    assert os.path.exists(path), f"Not found: {path}"

with zipfile.ZipFile(ZIPS[0]) as z:
    top = {n.split("/")[0] for n in z.namelist() if n}
nested = top == {"Bangla-Multimodal-FakeNews"}
if not os.path.isdir(os.path.join(PROJECT, "app")):
    os.makedirs(PROJECT, exist_ok=True)
    with zipfile.ZipFile(ZIPS[0]) as z:
        z.extractall("/content" if nested else PROJECT)

for path in ZIPS[1:]:
    with zipfile.ZipFile(path) as z:
        z.extractall(PROJECT)
print("All zips applied")

All zips applied


In [2]:
P = "/content/Bangla-Multimodal-FakeNews"
def has(rel, text):
    with open(f"{P}/{rel}", encoding="utf-8") as f:
        return text in f.read()

checks = {
    "evidence_app.py exists": os.path.exists(f"{P}/src/evidence_app.py"),
    "script 09 exists": os.path.exists(f"{P}/scripts/09_evidence_gate.py"),
    "OCR filter (src/ocr.py)": has("src/ocr.py", "def clean_ocr_boxes"),
    "checkbox in app": has("app/capstone_app.py", "শুধু মূল লেখা রাখুন"),
    "Evidence tab in app": has("app/capstone_app.py", "10. Evidence (Wikipedia)"),
    "base has image-only code": has("src/serving.py", "def image_only_probs"),
}
for name, ok in checks.items():
    print(("OK   " if ok else "MISS "), name)

OK    evidence_app.py exists
OK    script 09 exists
OK    OCR filter (src/ocr.py)
OK    checkbox in app
OK    Evidence tab in app
OK    base has image-only code


In [3]:
%cd /content/Bangla-Multimodal-FakeNews
from app.capstone_app import load_runtime, build_ui
load_runtime()
demo = build_ui()
demo.launch(share=False, debug=True)

/content/Bangla-Multimodal-FakeNews


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] ElectraModel LOAD REPORT from: csebuetnlp/banglabert
Key                                               | Status     |  | 
--------------------------------------------------+------------+--+-
discriminator_predictions.dense_prediction.bias   | UNEXPECTED |  | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED |  | 
discriminator_predictions.dense.bias              | UNEXPECTED |  | 
discriminator_predictions.dense.weight            | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>

/usr/local/lib/python3.13/dist-packages/torch/ao/nn/quantized/dynamic/modules/rnn.py:162: UserWarning: torch.quantize_per_tensor, torch.quantize_per_channel and other quantized tensor creation functions that produce tensors with dtype torch.quint8, torch.qint8, and torch.qint32 are deprecated and will be removed in a future PyTorch release. Please see https://github.com/pytorch/pytorch/issues/184982 for more information. (Triggered internally at /__w/pytorch/pytorch/aten/src/ATen/quantized/Quantizer.cpp:111.)
  w_ih = torch.quantize_per_tensor(
/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Keyboard interruption in main thread... closing server.
